In [1]:
!rm -rf Classical-Chinese-Summarization
!git clone https://github.com/ctaiyi15/Classical-Chinese-Summarization.git
!ls Classical-Chinese-Summarization/data

Cloning into 'Classical-Chinese-Summarization'...
remote: Enumerating objects: 7609, done.
remote: Counting objects: 100% (2225/2225), done.
remote: Compressing objects: 100% (2187/2187), done.
remote: Total 7609 (delta 488), reused 1764 (delta 36), pack-reused 5384 (from 1)
Receiving objects: 100% (7609/7609), 67.53 MiB | 7.35 MiB/s, done.
Resolving deltas: 100% (1310/1310), done.
Updating files: 100% (3777/3777), done.
dataset.json  processed  raw  segmented  segmented_v2


In [2]:
!ls Classical-Chinese-Summarization/data/processed
!pip install -U openai nest_asyncio
import asyncio
import json
import re
from pathlib import Path

import nest_asyncio
from openai import AsyncOpenAI
from google.colab import userdata
from tqdm.notebook import tqdm

input_root = Path(
    "Classical-Chinese-Summarization/data/processed/translated"
)

summary_root = Path(
    "Classical-Chinese-Summarization/data/processed/summary_clean"
)

evaluation  sample  summary  summary_clean  translated	translated_back


In [3]:
nest_asyncio.apply()

client = AsyncOpenAI(
    api_key=userdata.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

MAX_CONCURRENT_REQUESTS = 20
SEM = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

In [4]:
def safe_json_loads(content, debug_info=None):

    try:
        return json.loads(content)

    except json.JSONDecodeError as original_error:

        raw_content = content

        content = content.strip()

        content = re.sub(r"^```json", "", content)
        content = re.sub(r"^```", "", content)
        content = re.sub(r"```$", "", content)

        match = re.search(
            r"\{.*\}",
            content,
            re.DOTALL
        )

        if match:
            try:
                return json.loads(match.group())

            except json.JSONDecodeError as extracted_error:

                print("\n" + "=" * 80)
                print("JSON ERROR AFTER EXTRACTING JSON-LIKE OBJECT")
                print("=" * 80)

                if debug_info is not None:
                    print("DEBUG INFO:")
                    print(
                        json.dumps(
                            debug_info,
                            ensure_ascii=False,
                            indent=2
                        )
                    )

                print("\nRAW MODEL OUTPUT:")
                print(raw_content)
                print("=" * 80)

                raise extracted_error

        print("\n" + "=" * 80)
        print("JSON ERROR: NO VALID JSON OBJECT FOUND")
        print("=" * 80)

        if debug_info is not None:
            print("DEBUG INFO:")
            print(
                json.dumps(
                    debug_info,
                    ensure_ascii=False,
                    indent=2
                )
            )

        print("\nRAW MODEL OUTPUT:")
        print(raw_content)
        print("=" * 80)

        raise original_error

async def guarded_completion(messages, temperature=0, max_tokens=10000):

    async with SEM:

        response = await client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            temperature=temperature,
            response_format={"type": "json_object"},
            max_tokens=max_tokens
        )

    return response

def build_pairs():

    pairs = []

    source_files = list(input_root.rglob("*.txt"))

    for src_path in source_files:

        rel = src_path.relative_to(input_root)

        summary_path = (
            summary_root / rel
        ).with_suffix(".summary.txt")

        if summary_path.exists():
            pairs.append((src_path, summary_path))

    return pairs


def read_file(path):

    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

In [5]:
# Coverage/Recall Evaluation
# 1. Extract 20 important points from the original text;
# 2. Check whether each point is included in the summary

COVERAGE_SAVE_PATH = "coverage_live_results.json"
COVERAGE_FAILED_SAVE_PATH = "coverage_failed_files.json"

def load_important_points_json(content, debug_info=None):

    raw_content = content

    # First try normal JSON parsing.
    try:
        return safe_json_loads(
            content,
            debug_info=debug_info
        )["important_points"]

    except json.JSONDecodeError:
        pass

    # Clean markdown fences.
    text = content.strip()
    text = re.sub(r"^```json", "", text)
    text = re.sub(r"^```", "", text)
    text = re.sub(r"```$", "", text)
    text = text.strip()

    # Try simple repairs for missing closing characters.
    repair_candidates = [
        text,
        text.rstrip(",") + "\n  ]\n}",
        text.rstrip(",") + "\n]",
        text.rstrip(",") + "\n}"
    ]

    for candidate in repair_candidates:
        try:
            obj = json.loads(candidate)
            if "important_points" in obj:
                return obj["important_points"]
        except json.JSONDecodeError:
            continue

    # Last-resort salvage:
    # Extract quoted strings inside the important_points array.
    array_start = text.find("[")
    if array_start != -1:
        array_text = text[array_start + 1:]

        string_matches = re.findall(
            r'"((?:\\.|[^"\\])*)"',
            array_text,
            flags=re.DOTALL
        )

        points = []

        for s in string_matches:
            try:
                points.append(
                    json.loads(f'"{s}"')
                )
            except json.JSONDecodeError:
                points.append(s)

        if points:
            print("\n" + "=" * 80)
            print("WARNING: Salvaged important_points from malformed JSON")
            print("=" * 80)

            if debug_info is not None:
                print("DEBUG INFO:")
                print(
                    json.dumps(
                        debug_info,
                        ensure_ascii=False,
                        indent=2
                    )
                )

            print("Recovered points:", len(points))
            print("=" * 80)

            return points

    print("\n" + "=" * 80)
    print("FAILED TO PARSE OR SALVAGE important_points JSON")
    print("=" * 80)

    if debug_info is not None:
        print("DEBUG INFO:")
        print(
            json.dumps(
                debug_info,
                ensure_ascii=False,
                indent=2
            )
        )

    print("\nRAW OUTPUT:")
    print(raw_content)
    print("=" * 80)

    raise json.JSONDecodeError(
        "Could not parse important_points JSON",
        raw_content,
        0
    )

async def extract_important_points(source_text, debug_info=None):

    prompt = f"""
You are an expert evaluator of historical summaries.

Task:
Read the original historical text and identify the 20 most important factual points that a good summary should include.

Rules:
- Return exactly 20 points.
- Each point must express one important factual idea.
- Do NOT include minor details unless they are central to the text.
- Do NOT add outside knowledge.
- Use concise English.
- Preserve names and historical entities when important.

Return JSON ONLY:
{{
  "important_points": [
    "point 1",
    "point 2"
  ]
}}

ORIGINAL TEXT:
{source_text}
"""

    response = await guarded_completion(
        [{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=5000
    )

    choice = response.choices[0]
    content = choice.message.content

    print("extract_important_points finish_reason:", choice.finish_reason)

    return load_important_points_json(
        content,
        debug_info=debug_info
    )[:20]


async def check_point_in_summary(point, summary_text, debug_info=None):

    prompt = f"""
You are evaluating summary coverage.

Task:
Determine whether the summary includes the important point.

Rules:
- ONLY use the summary.
- Judge meaning, not exact wording.
- Paraphrases count as included if the meaning is the same.
- Minor tense/aspect differences are acceptable.
- If the summary includes only part of the point, label it "partially_included".
- If the summary does not include the point, label it "not_included".
- Be strict about factual content, but not about wording.

Return JSON ONLY:
{{
  "label": "included"
}}

Allowed labels:
- "included"
- "partially_included"
- "not_included"

IMPORTANT POINT:
{point}

SUMMARY:
{summary_text}
"""

    response = await guarded_completion(
        [{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=100
    )

    content = response.choices[0].message.content

    return safe_json_loads(
        content,
        debug_info=debug_info
    )


async def process_coverage_point(point, summary_text, debug_info=None):

    verdict = await check_point_in_summary(
        point,
        summary_text,
        debug_info=debug_info
    )

    return {
        "point": point,
        "label": verdict["label"]
    }


async def evaluate_coverage(source_text, summary_text, debug_info=None):

    important_points = await extract_important_points(
        source_text,
        debug_info={
            **(debug_info or {}),
            "stage": "extract_important_points"
        }
    )

    # Defensive cleanup
    important_points = important_points[:20]

    tasks = [
        process_coverage_point(
            point,
            summary_text,
            debug_info={
                **(debug_info or {}),
                "stage": "check_point_in_summary",
                "point_index": idx,
                "total_points": len(important_points),
                "point": point
            }
        )
        for idx, point in enumerate(important_points, start=1)
    ]

    results = await asyncio.gather(*tasks)

    included = sum(
        r["label"] == "included"
        for r in results
    )

    partially_included = sum(
        r["label"] == "partially_included"
        for r in results
    )

    not_included = sum(
        r["label"] == "not_included"
        for r in results
    )

    total = len(results)

    coverage_score = (
        included + 0.5 * partially_included
    ) / total if total > 0 else 0

    return {
        "coverage_score": coverage_score,
        "total_points": total,
        "included_points": included,
        "partially_included_points": partially_included,
        "not_included_points": not_included,
        "details": results
    }


async def process_file_coverage(src_path, sum_path):

    source_text = read_file(src_path)
    summary_text = read_file(sum_path)

    debug_info = {
        "source_file": str(src_path),
        "summary_file": str(sum_path)
    }

    result = await evaluate_coverage(
        source_text,
        summary_text,
        debug_info=debug_info
    )

    return {
        "source_file": str(src_path),
        "summary_file": str(sum_path),
        **result
    }


async def process_file_coverage_with_paths(src, summ):

    try:
        result = await process_file_coverage(src, summ)

        return {
            "ok": True,
            "source_file": str(src),
            "summary_file": str(summ),
            "result": result
        }

    except Exception as e:

        return {
            "ok": False,
            "source_file": str(src),
            "summary_file": str(summ),
            "error_type": type(e).__name__,
            "error": str(e),
            "repr": repr(e)
        }


async def run_first_x_coverage_live_saved(x):

    pairs = build_pairs()[:x]

    print(f"Running coverage evaluation on {len(pairs)} file pairs")

    results = []
    errors = []

    tasks = [
        process_file_coverage_with_paths(src, summ)
        for src, summ in pairs
    ]

    completed = 0
    failed = 0

    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks)):

        item = await coro

        if item["ok"]:

            result = item["result"]
            results.append(result)
            completed += 1

            score = result["coverage_score"]

            avg = sum(
                r["coverage_score"]
                for r in results
            ) / len(results)

            print("\n" + "=" * 60)
            print(f"SUCCESS [{completed}/{len(tasks)}]")
            print("Source file:", item["source_file"])
            print("Summary file:", item["summary_file"])
            print("Coverage score:", score)
            print("Running avg:", avg)

            display_result = {
                k: v
                for k, v in result.items()
                if k != "details"
            }

            print(
                json.dumps(
                    display_result,
                    ensure_ascii=False,
                    indent=2
                )
            )

            with open(
                COVERAGE_SAVE_PATH,
                "w",
                encoding="utf-8"
            ) as f:
                json.dump(
                    results,
                    f,
                    ensure_ascii=False,
                    indent=2
                )

        else:

            failed += 1
            errors.append(item)

            print("\n" + "=" * 60)
            print(f"FAILED [{failed}/{len(tasks)}]")
            print("Source file:", item["source_file"])
            print("Summary file:", item["summary_file"])
            print("ERROR:", item["repr"])

            with open(
                COVERAGE_FAILED_SAVE_PATH,
                "w",
                encoding="utf-8"
            ) as f:
                json.dump(
                    errors,
                    f,
                    ensure_ascii=False,
                    indent=2
                )

    print("\nDONE")
    print("Successful files:", len(results))
    print("Failed files:", len(errors))

    if results:
        final_avg = sum(
            r["coverage_score"]
            for r in results
        ) / len(results)

        print("Final coverage avg:", final_avg)
    else:
        print("No successful results. Final avg cannot be computed.")

    if errors:
        print("Failed file list saved to:", COVERAGE_FAILED_SAVE_PATH)

    return results


async def run_first_coverage_debug():

    pairs = build_pairs()[:1]

    print(f"Running coverage debug on {len(pairs)} file pair")

    if not pairs:
        print("No file pairs found")
        return None

    src_path, sum_path = pairs[0]

    print("Source file:", src_path)
    print("Summary file:", sum_path)

    result = await process_file_coverage(src_path, sum_path)

    display_result = {
        k: v
        for k, v in result.items()
        if k != "details"
    }

    print(
        json.dumps(
            display_result,
            ensure_ascii=False,
            indent=2
        )
    )

    return result

coverage_debug_result = await run_first_x_coverage_live_saved(1000)


Running coverage evaluation on 409 file pairs


  0%|          | 0/409 [00:00<?, ?it/s]

流式输出内容被截断，只能显示最后 5000 行内容。
  "partially_included_points": 0,
  "not_included_points": 3
}

SUCCESS [98/409]
Source file: Classical-Chinese-Summarization/data/processed/translated/史记/三十世家/魏世家/target.txt
Summary file: Classical-Chinese-Summarization/data/processed/summary_clean/史记/三十世家/魏世家/target.summary.txt
Coverage score: 0.7
Running avg: 0.8224489795918367
{
  "source_file": "Classical-Chinese-Summarization/data/processed/translated/史记/三十世家/魏世家/target.txt",
  "summary_file": "Classical-Chinese-Summarization/data/processed/summary_clean/史记/三十世家/魏世家/target.summary.txt",
  "coverage_score": 0.7,
  "total_points": 20,
  "included_points": 14,
  "partially_included_points": 0,
  "not_included_points": 6
}

SUCCESS [99/409]
Source file: Classical-Chinese-Summarization/data/processed/translated/后汉书/列传/蔡邕列传下/target.txt
Summary file: Classical-Chinese-Summarization/data/processed/summary_clean/后汉书/列传/蔡邕列传下/target.summary.txt
Coverage score: 0.9
Running avg: 0.8232323232323232
{
  "source_file"

In [6]:
from google.colab import files

files.download("coverage_live_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>